# 05 — Whole-Slide Image Validation

Test whether the transcriptomically defined temporal transition is independently
detectable in routine diagnostic histopathology.

Pipeline:
1. Scan whole-slide images per patient and extract tissue patches
2. Encode each patch with a frozen ResNet50 (ImageNet pretrained)
3. Mean-pool patch features to patient-level embeddings
4. PCA on patient embeddings
5. Multivariate Cox: phase predictor + PC1 covariate, overall survival outcome,
   stratified by cancer type

**Why ResNet50 with ImageNet weights rather than a pathology foundation model:**
Current pathology foundation models (UNI, CONCH, Virchow) are pretrained on TCGA,
including the slides analyzed here. ImageNet-pretrained ResNet50 keeps the
morphological signal independent of TCGA-derived representations and provides a
conservative baseline for the temporal transition.


In [ ]:
import json
import os
import pickle
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm

warnings.filterwarnings("ignore")

try:
    import openslide
except ImportError:
    raise ImportError("openslide-python required: pip install openslide-python")

from lifelines import CoxPHFitter
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import joblib

BASE_DIR = Path(os.environ.get("COSMOS_BASE_DIR", "./data"))
RAW_DIR = BASE_DIR / "Raw Data"
INTER_DIR = BASE_DIR / "intermediates"
WSI_DIR = BASE_DIR / "wsi_results"
WSI_DIR.mkdir(parents=True, exist_ok=True)
for sub in ["features", "results", "cache"]:
    (WSI_DIR / sub).mkdir(exist_ok=True, parents=True)

DAYS_PER_MO = 365.25 / 12
LANDMARK_M = 24
MIN_OS_EVENTS = 15


## Configuration


In [ ]:
CDR_FILE = RAW_DIR / "tcga_clinical" / "TCGA-CDR.csv"
if not CDR_FILE.exists():
    CDR_FILE = CDR_FILE.with_suffix(".xlsx")

WSI_DATA_DIR = Path(os.environ.get("COSMOS_WSI_DIR", BASE_DIR / "WSI_slides"))
CANCER_TYPES = ["BRCA", "LUAD", "LUSC", "KIRC", "LIHC", "STAD", "COAD", "HNSC", "UCEC"]
MAX_PATCHES = 300
BATCH_SIZE = 128
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"WSI directory: {WSI_DATA_DIR}")


## Clinical loading

Same phase-predictor logic as notebook 04: DFI preferred, PFI as fallback. Both
recurrence endpoint and OS endpoint are retained per patient.


In [ ]:
def extract_patient_barcode(filename):
    """Extract TCGA-XX-YYYY patient barcode from a slide filename."""
    parts = str(filename).split("-")
    if len(parts) >= 3 and parts[0] == "TCGA":
        return f"{parts[0]}-{parts[1]}-{parts[2]}"
    return None


def load_clinical_data(cdr_file, cancer_types):
    """Load CDR with phase predictor (DFI preferred, PFI fallback) and OS outcome."""
    cdr_file = Path(cdr_file)
    cdr = (pd.read_excel(cdr_file) if cdr_file.suffix.lower() == ".xlsx"
           else pd.read_csv(cdr_file))
    cdr = cdr.rename(columns={"bcr_patient_barcode": "patient_id", "type": "cancer_type"})

    cdr["os_time"] = pd.to_numeric(cdr["OS.time"], errors="coerce") / DAYS_PER_MO
    cdr["os_event"] = (pd.to_numeric(cdr["OS"], errors="coerce") > 0).astype(float)

    has_dfi = "DFI.time" in cdr.columns and "DFI" in cdr.columns
    has_pfi = "PFI.time" in cdr.columns and "PFI" in cdr.columns
    if not has_dfi and not has_pfi:
        raise ValueError("CDR needs at least one of DFI or PFI")

    if has_dfi:
        cdr["dfi_time"] = pd.to_numeric(cdr["DFI.time"], errors="coerce") / DAYS_PER_MO
        cdr["dfi_event"] = (pd.to_numeric(cdr["DFI"], errors="coerce") > 0).astype(float)
    else:
        cdr["dfi_time"] = np.nan
        cdr["dfi_event"] = np.nan
    if has_pfi:
        cdr["pfi_time"] = pd.to_numeric(cdr["PFI.time"], errors="coerce") / DAYS_PER_MO
        cdr["pfi_event"] = (pd.to_numeric(cdr["PFI"], errors="coerce") > 0).astype(float)
    else:
        cdr["pfi_time"] = np.nan
        cdr["pfi_event"] = np.nan

    cdr["rec_time"] = cdr["dfi_time"].where(~cdr["dfi_time"].isna(), cdr["pfi_time"])
    cdr["rec_event"] = cdr["dfi_event"].where(~cdr["dfi_event"].isna(), cdr["pfi_event"])

    cdr = cdr[cdr["cancer_type"].isin(cancer_types)].copy()
    cdr = cdr.dropna(subset=["patient_id", "cancer_type", "os_time", "os_event", "rec_time", "rec_event"])
    cdr = cdr[(cdr["os_time"] > 0) & (cdr["rec_time"] > 0)]
    cdr["phase_early"] = ((cdr["rec_event"] == 1) & (cdr["rec_time"] <= LANDMARK_M)).astype(int)

    keep = ["patient_id", "cancer_type", "os_time", "os_event",
            "rec_time", "rec_event", "phase_early"]
    return cdr[keep].drop_duplicates("patient_id")


cdr_df = load_clinical_data(CDR_FILE, CANCER_TYPES)
print(f"Clinical: n={len(cdr_df)}, early phase={int(cdr_df['phase_early'].sum())}")
for ct in sorted(CANCER_TYPES):
    sub = cdr_df[cdr_df["cancer_type"] == ct]
    if len(sub) > 0:
        print(f"  {ct}: n={len(sub)}, OS events={int(sub['os_event'].sum())}, "
              f"early={int(sub['phase_early'].sum())}")


## Slide scanning


In [ ]:
SLIDE_EXTS = [".svs", ".tif", ".tiff", ".ndpi", ".scn", ".mrxs", ".vms", ".vmu"]


def scan_slides(data_dir, cancer_types):
    """Find slides under data_dir. Tries layout A first (cancer subfolders),
    falls back to layout B (flat directory)."""
    data_path = Path(data_dir)
    if not data_path.exists():
        raise FileNotFoundError(f"slide directory not found: {data_path}")

    all_slides = []
    for cancer_type in cancer_types:
        folder = data_path / cancer_type
        if not folder.exists():
            continue
        for ext in SLIDE_EXTS:
            for fp in folder.rglob(f"*{ext}"):
                pid = extract_patient_barcode(fp.name)
                if pid:
                    all_slides.append({
                        "slide_id": fp.stem,
                        "slide_path": str(fp),
                        "patient_id": pid,
                        "cancer_type_hint": cancer_type,
                    })

    if not all_slides:
        for ext in SLIDE_EXTS:
            for fp in data_path.rglob(f"*{ext}"):
                pid = extract_patient_barcode(fp.name)
                if pid:
                    all_slides.append({
                        "slide_id": fp.stem,
                        "slide_path": str(fp),
                        "patient_id": pid,
                        "cancer_type_hint": fp.parent.name,
                    })

    df = pd.DataFrame(all_slides).drop_duplicates("slide_path")
    print(f"Found {len(df)} unique slides")
    return df


slides_df = scan_slides(WSI_DATA_DIR, CANCER_TYPES)


## Feature extraction


In [ ]:
def load_feature_extractor(device):
    """ResNet50 with ImageNet weights, fc layer replaced by identity to produce
    2048-dim embeddings."""
    try:
        from torchvision.models import resnet50, ResNet50_Weights
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    except Exception:
        from torchvision import models
        model = models.resnet50(pretrained=True)
    model.fc = nn.Identity()
    return model.to(device).eval()


def get_image_transform():
    from torchvision import transforms
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])


def extract_patches_from_slide(slide_path, max_patches=300, patch_size=224, force_dense=False):
    """Grid sample patches from a slide with a simple background filter."""
    try:
        slide = openslide.OpenSlide(str(slide_path))
        level = min(2, slide.level_count - 1)
        width, height = slide.level_dimensions[level]
        downsample = slide.level_downsamples[level]
        if width <= patch_size or height <= patch_size:
            slide.close()
            return []
        base_stride = int(np.sqrt(width * height / max(1, max_patches)))
        stride = max(256 if force_dense else 512, base_stride)

        patches = []
        for y in range(0, height - patch_size, stride):
            for x in range(0, width - patch_size, stride):
                if len(patches) >= max_patches:
                    break
                try:
                    patch = slide.read_region(
                        (int(x * downsample), int(y * downsample)),
                        level, (patch_size, patch_size),
                    ).convert("RGB")
                    thumb = np.array(patch.resize((32, 32)))
                    if thumb.mean() < 220:
                        patches.append(patch)
                except Exception:
                    continue
            if len(patches) >= max_patches:
                break
        slide.close()
        return patches
    except Exception:
        return []


def extract_features_from_patches(patches, model, transform, device, batch_size=128):
    """Encode patches to 2048-dim ResNet50 features."""
    all_features = []
    with torch.no_grad():
        for i in range(0, len(patches), batch_size):
            batch = patches[i:i + batch_size]
            batch_tensor = torch.stack([transform(p) for p in batch]).to(device)
            feats = model(batch_tensor)
            all_features.append(feats.detach().cpu().numpy())
            del batch_tensor, feats
            if device.type == "cuda":
                torch.cuda.empty_cache()
    return np.vstack(all_features) if all_features else None


model = load_feature_extractor(device)
transform = get_image_transform()


## Per-slide processing


In [ ]:
def process_cohort(slides_df, cdr_df, model, transform, device,
                   max_patches, batch_size, output_dir):
    """Extract features for all slides with matching clinical data."""
    cohort_df = slides_df.merge(
        cdr_df[["patient_id", "cancer_type", "os_time", "os_event",
                "rec_time", "rec_event", "phase_early"]],
        on="patient_id", how="inner",
    )
    print(f"Slides with clinical data: {len(cohort_df)} of {len(slides_df)}")

    processed_slides = []
    slide_embeddings = []
    for _, row in tqdm(cohort_df.iterrows(), total=len(cohort_df), desc="Slides"):
        slide_path = Path(row["slide_path"])
        if not slide_path.exists():
            continue
        patches = extract_patches_from_slide(
            slide_path, max_patches=max_patches, patch_size=224, force_dense=False,
        )
        if len(patches) < 10:
            patches = extract_patches_from_slide(
                slide_path, max_patches=max_patches, patch_size=224, force_dense=True,
            )
        if len(patches) < 10:
            continue
        features = extract_features_from_patches(
            patches, model, transform, device, batch_size=batch_size,
        )
        if features is None or len(features) == 0:
            continue
        slide_embeddings.append(np.mean(features, axis=0))
        processed_slides.append({
            "slide_id": row["slide_id"],
            "slide_path": str(slide_path),
            "patient_id": row["patient_id"],
            "cancer_type": row["cancer_type"],
            "os_time": float(row["os_time"]),
            "os_event": float(row["os_event"]),
            "rec_time": float(row["rec_time"]),
            "rec_event": float(row["rec_event"]),
            "phase_early": int(row["phase_early"]),
            "n_patches": int(len(patches)),
        })

    processed_df = pd.DataFrame(processed_slides)
    processed_df.to_csv(output_dir / "results" / "processed_slides.csv", index=False)
    if not slide_embeddings:
        raise RuntimeError("no slide embeddings produced")
    embeddings_array = np.vstack(slide_embeddings)
    np.save(output_dir / "features" / "slide_embeddings.npy", embeddings_array)
    print(f"Processed {len(processed_df)} slides, embeddings shape {embeddings_array.shape}")
    return processed_df, embeddings_array


processed_df, embeddings_array = process_cohort(
    slides_df, cdr_df, model, transform, device,
    MAX_PATCHES, BATCH_SIZE, WSI_DIR,
)


## Patient-level aggregation and PCA


In [ ]:
def aggregate_to_patient_level(processed_df, embeddings_array):
    """Mean-pool slide embeddings to patient-level."""
    if processed_df is None or len(processed_df) == 0:
        raise RuntimeError("nothing to aggregate")
    for i in range(embeddings_array.shape[1]):
        processed_df[f"emb_{i}"] = embeddings_array[:, i]
    emb_cols = [f"emb_{i}" for i in range(embeddings_array.shape[1])]

    patient_df = (
        processed_df.groupby("patient_id")
        .agg(
            cancer_type=("cancer_type", "first"),
            os_time=("os_time", "first"),
            os_event=("os_event", "first"),
            rec_time=("rec_time", "first"),
            rec_event=("rec_event", "first"),
            phase_early=("phase_early", "first"),
            n_slides=("slide_id", "count"),
            **{c: (c, "mean") for c in emb_cols},
        )
        .reset_index()
    )
    return patient_df, patient_df[emb_cols].values


def perform_pca(patient_embeddings):
    """Standardize and run PCA on patient embeddings."""
    scaler = StandardScaler()
    X = scaler.fit_transform(patient_embeddings)
    n_comp = min(10, X.shape[0], X.shape[1])
    pca = PCA(n_components=n_comp, random_state=42)
    pcs = pca.fit_transform(X)
    evr = pca.explained_variance_ratio_
    print(f"PCA variance: PC1={evr[0]:.1%}, PC2={evr[1]:.1%}, PC3={evr[2]:.1%}")
    return pcs, pca, scaler


patient_df, patient_embeddings = aggregate_to_patient_level(processed_df, embeddings_array)
pcs, pca, scaler = perform_pca(patient_embeddings)
print(f"Patient-level: n={len(patient_df)}")


## Cox regression

Multivariate Cox with `phase_early` (binary) and PC1 (continuous) as covariates,
overall survival as outcome, stratified by cancer type to account for cross-cancer
baseline hazard differences.


In [ ]:
def perform_cox_regression(patient_df, pcs, strata_pan_cancer=True):
    """Multivariate Cox: phase_early + PC1 -> OS, stratified by cancer type."""
    df = patient_df.copy()
    df["PC1"] = pcs[:, 0]
    if pcs.shape[1] > 1:
        df["PC2"] = pcs[:, 1]

    # Sign-correct PC1 so higher values associate with early phase (convention only)
    try:
        corr = pd.Series(df["PC1"]).corr(pd.Series(df["phase_early"]))
        if pd.notna(corr) and corr < 0:
            df["PC1"] *= -1
    except Exception:
        pass

    cox_data = df[["patient_id", "cancer_type", "os_time", "os_event",
                    "phase_early", "PC1"]].dropna()
    cox_data = cox_data[cox_data["os_event"].isin([0., 1.])]

    cph = CoxPHFitter(penalizer=0.1)
    if strata_pan_cancer:
        cph.fit(
            cox_data[["os_time", "os_event", "phase_early", "PC1", "cancer_type"]],
            duration_col="os_time", event_col="os_event",
            strata=["cancer_type"],
        )
    else:
        cph.fit(
            cox_data[["os_time", "os_event", "phase_early", "PC1"]],
            duration_col="os_time", event_col="os_event",
        )

    hr_phase = float(np.exp(cph.params_["phase_early"]))
    hr_pc1 = float(np.exp(cph.params_["PC1"]))
    p_phase = float(cph.summary.loc["phase_early", "p"])
    p_pc1 = float(cph.summary.loc["PC1", "p"])
    ci_l_ph = float(np.exp(cph.confidence_intervals_.loc["phase_early", "95% lower-bound"]))
    ci_u_ph = float(np.exp(cph.confidence_intervals_.loc["phase_early", "95% upper-bound"]))
    c_index = float(cph.concordance_index_)

    print(f"Pan-cancer (stratified):")
    print(f"  n={len(cox_data)}, OS events={int(cox_data['os_event'].sum())}")
    print(f"  phase_early: HR={hr_phase:.2f} [{ci_l_ph:.2f}-{ci_u_ph:.2f}], P={p_phase:.2e}")
    print(f"  PC1:         HR={hr_pc1:.2f}, P={p_pc1:.2e}")
    print(f"  C-index: {c_index:.3f}")

    # Per-cancer
    cancer_results = []
    print(f"\nPer-cancer Cox:")
    print(f"  {'Cancer':<8} {'n':>5} {'ev':>4} {'early':>6}  {'HR':>6}  {'95% CI':>15}  {'P':>10}  {'C':>6}")
    for ct in sorted(cox_data["cancer_type"].unique()):
        subset = cox_data[cox_data["cancer_type"] == ct].copy()
        n_ev = int(subset["os_event"].sum())
        n_early = int(subset["phase_early"].sum())
        if n_ev < MIN_OS_EVENTS or n_early < 3 or subset["phase_early"].nunique() < 2:
            print(f"  {ct:<8} {len(subset):>5} {n_ev:>4} {n_early:>6}  skipped")
            continue
        try:
            cph_ct = CoxPHFitter(penalizer=0.1)
            cph_ct.fit(
                subset[["os_time", "os_event", "phase_early", "PC1"]],
                duration_col="os_time", event_col="os_event",
            )
            hr = float(np.exp(cph_ct.params_["phase_early"]))
            ci_l = float(np.exp(cph_ct.confidence_intervals_.loc["phase_early", "95% lower-bound"]))
            ci_u = float(np.exp(cph_ct.confidence_intervals_.loc["phase_early", "95% upper-bound"]))
            p = float(cph_ct.summary.loc["phase_early", "p"])
            hr_p1 = float(np.exp(cph_ct.params_["PC1"]))
            conc = float(cph_ct.concordance_index_)
            cancer_results.append({
                "Cancer": ct, "N": len(subset), "N_early": n_early, "Events": n_ev,
                "HR_phase": hr, "CI_Lower": ci_l, "CI_Upper": ci_u,
                "P_phase": p, "HR_PC1": hr_p1, "C_index": conc,
            })
            print(f"  {ct:<8} {len(subset):>5} {n_ev:>4} {n_early:>6}  "
                  f"{hr:>6.2f}  [{ci_l:.2f}-{ci_u:.2f}]  {p:>10.4f}  {conc:>6.3f}")
        except Exception as e:
            print(f"  {ct}: failed ({e})")

    return {
        "N_Patients": int(len(cox_data)),
        "N_Events": int(cox_data["os_event"].sum()),
        "Phase_HR": hr_phase,
        "Phase_CI_Lower": ci_l_ph,
        "Phase_CI_Upper": ci_u_ph,
        "Phase_P": p_phase,
        "PC1_HR": hr_pc1,
        "PC1_P": p_pc1,
        "CIndex": c_index,
        "Timestamp": datetime.now().isoformat(),
        "Method": (
            "Phase from DFI (PFI fallback), landmark 24m. Cox outcome OS. "
            "PC1 from ResNet50 ImageNet PCA. penalizer=0.1. "
            "Pan-cancer stratified by cancer_type."
        ),
    }, pd.DataFrame(cancer_results), cox_data


summary, cancer_df, cox_data = perform_cox_regression(patient_df, pcs, strata_pan_cancer=True)


## Save outputs


In [ ]:
joblib.dump(pca, INTER_DIR / "wsi_pca_model.pkl")
joblib.dump(scaler, INTER_DIR / "wsi_scaler.pkl")

patient_df.to_csv(WSI_DIR / "patient_level_data.csv", index=False)
cancer_df.to_csv(WSI_DIR / "cancer_specific_results.csv", index=False)
cox_data.to_csv(WSI_DIR / "cox_data.csv", index=False)

with open(WSI_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("WSI validation complete")
print(f"Outputs in: {WSI_DIR}")
